In [1]:
import pandas as pd
import openpyxl
import os

In [2]:
# 1. Configuración de Rutas y Estructuras
CATALOG_PATH = "../DICCIONARIOS/240708 Catalogos.xlsx"
FILES_MADE = [
    "../DATA/Silver/COVID19MEXICO2021/COVID19MEXICO2021.csv",
    "../DATA/Silver/COVID19MEXICO2022/COVID19MEXICO2022.csv",
    "../DATA/Silver/COVID19MEXICO2023/COVID19MEXICO2023.csv",
    "../DATA/Silver/COVID19MEXICO2024/COVID19MEXICO2024.csv"
]

# Definición de Clústeres (DIMENSIONS)
DIMENSIONS = {
    'DIM_Geografico_informacion_paciente': ['ENTIDAD_UM', 'ENTIDAD_NAC'],
    'DIM_Geografico_residencia': ['ENTIDAD_RES', 'MUNICIPIO_RES'],
    'DIM_Geografico_Nacionalidad': ['NACIONALIDAD', 'PAIS_NACIONALIDAD', 'PAIS_ORIGEN'],
    'DIM_Descripcion_del_paciente': ['SEXO', 'EDAD', 'TIPO_PACIENTE'],
    'DIM_Indigena': ['HABLA_LENGUA_INDIG', 'INDIGENA', 'MIGRANTE'],
    'DIM_Comorbilidades_Respiratorias': ['INTUBADO', 'NEUMONIA', 'EPOC', 'ASMA', 'TABAQUISMO'],
    'DIM_Comorbilidades_de_presion': ['DIABETES', 'INMUSUPR', 'HIPERTENSION', 'CARDIOVASCULAR', 'OBESIDAD'],
    'DIM_Otras_caracteristicas_medicas': ['EMBARAZO', 'RENAL_CRONICA', 'OTRA_COM'],
    'DIM_Ubicacion_de_laboratorio': ['ORIGEN', 'SECTOR', 'OTRO_CASO', 'UCI', 'RESULTADO_PCR', 'RESULTADO_PCR_COINFECCION'], # PCR omitido si no existe en data
    'DIM_Antigeno': ['TOMA_MUESTRA_ANTIGENO', 'RESULTADO_ANTIGENO'],
    'DIM_Datos_de_laboratorio': ['TOMA_MUESTRA_LAB', 'RESULTADO_LAB', 'CLASIFICACION_FINAL_COVID', 'CLASIFICACION_FINAL_FLU']
}

# Mapeo de Columna a sheet de Excel
CATALOG_MAP = {
    'ORIGEN': 'ORIGEN', 'SECTOR': 'SECTOR', 'ENTIDAD_UM': 'ENTIDADES',
    'SEXO': 'SEXO', 'ENTIDAD_NAC': 'ENTIDADES', 'ENTIDAD_RES': 'ENTIDADES',
    'MUNICIPIO_RES': 'MUNICIPIOS', 'TIPO_PACIENTE': 'TIPO_PACIENTE',
    'INTUBADO': 'SI_ NO', 'NEUMONIA': 'SI_ NO', 'NACIONALIDAD': 'NACIONALIDAD',
    'EMBARAZO': 'SI_ NO', 'HABLA_LENGUA_INDIG': 'SI_ NO', 'INDIGENA': 'SI_ NO',
    'DIABETES': 'SI_ NO', 'EPOC': 'SI_ NO', 'ASMA': 'SI_ NO', 'INMUSUPR': 'SI_ NO',
    'HIPERTENSION': 'SI_ NO', 'OTRA_COM': 'SI_ NO', 'CARDIOVASCULAR': 'SI_ NO',
    'OBESIDAD': 'SI_ NO', 'RENAL_CRONICA': 'SI_ NO', 'TABAQUISMO': 'SI_ NO',
    'OTRO_CASO': 'SI_ NO', 'TOMA_MUESTRA_LAB': 'SI_ NO', 'RESULTADO_LAB': 'RESULTADO_LAB',
    'RESULTADO_PCR': 'RESULTADO_PCR', 'RESULTADO_PCR_COINFECCION': 'RESULTADO_PCR',
    'TOMA_MUESTRA_ANTIGENO': 'SI_ NO', 'RESULTADO_ANTIGENO': 'RESULTADO_ANTIGENO',
    'CLASIFICACION_FINAL_COVID': 'CLASIFICACION_FINAL_COVID',
    'CLASIFICACION_FINAL_FLU': 'CLASIFICACION_FINAL_FLU', 'MIGRANTE': 'SI_ NO', 'UCI': 'SI_ NO'
}

In [22]:
def catalog_load(ruta_excel):
    """Extrae las traducciones de todas las sheets del Excel dinámicamente."""
    xls = pd.ExcelFile(ruta_excel)
    diccionarys = {}
    for sheet in xls.sheet_names:
        df_sheet = pd.read_excel(xls, sheet_name=sheet)
        # Asume columna 0 = Clave, columna 1 = Descripción
        diccionarys[sheet] = dict(zip(df_sheet.iloc[:, 0], df_sheet.iloc[:, 1]))
    return diccionarys

In [23]:
def unique_convin_extract():
    """Itera sobre los CSV masivos en chunks y extrae combinations empíricas LIMPIAS."""
    global_convinations = {dim: [] for dim in DIMENSIONS.keys()}
    
    for file in FILES_MADE:
        if not os.path.exists(file): 
            continue
        print(f"Procesando: {file}...")
        
        lot_iterator = pd.read_csv(file, chunksize=250000, low_memory=False, encoding='latin1')
        for chunk in lot_iterator:
            for name_dim, columns in DIMENSIONS.items():
                # 1. Filtrar columnas que existan en el chunk
                columns_presents = [c for c in columns if c in chunk.columns]
                
                if columns_presents:
                    # 2. CORRECCIÓN CRÍTICA: Seleccionar SOLO las columnas originales definidas en DIMENSIONS
                    # Esto descarta automáticamente cualquier columna 'DESC_' que venga en el CSV original
                    df_unique = chunk[columns_presents].drop_duplicates()
                    
                    # 3. Doble seguridad: Eliminar cualquier columna residual que empiece con 'DESC_' 
                    # (Por si acaso alguna se coló en la definición de DIMENSIONS o el CSV tiene basura)
                    cols_finales = [c for c in df_unique.columns if not c.startswith('DESC_')]
                    df_unique = df_unique[cols_finales]
                    
                    global_convinations[name_dim].append(df_unique)
                    
    return global_convinations

In [28]:
def process_and_export_dimensions(global_convinations, dict_catalogos):
    """Consolida combinations, aplica traducciones y genera la tabla final."""
    if not os.path.exists("dimensiones"):
        os.makedirs("dimensiones")

    for name_dim, dfs_list in global_convinations.items():
        if not dfs_list: 
            continue
        
        # 1. Consolidación final de la dimensión (Ahora viene limpia desde el extract)
        final_dim = pd.concat(dfs_list).drop_duplicates().reset_index(drop=True)
        
        # 2. Mapeo de traducciones
        # Como los datos de entrada ya están limpios, esto generará exactamente una columna DESC_ por cada original
        for col in final_dim.columns:
            if col in CATALOG_MAP:
                name_sheet = CATALOG_MAP[col]
                traduction_dyc = dict_catalogos.get(name_sheet, {})
                
                col_desc = f'DESC_{col}'
                final_dim[col_desc] = final_dim[col].map(traduction_dyc)
                
                # Llenado de nulos/valores texto abiertos
                final_dim[col_desc] = final_dim[col_desc].fillna(final_dim[col].astype(str))

        # 3. Generación de Llave Sustituta
        final_dim.insert(0, f'ID_{name_dim.upper()}', range(1, len(final_dim) + 1))
        
        # 4. Exportación
        exit_path = f"dimensiones/{name_dim}.csv"
        cols_to_drop = [c for c in final_dim.columns if 'DESC_' in c]
        final_dim = final_dim.drop(columns=cols_to_drop)
        final_dim.to_csv(exit_path, index=False, encoding='utf-8')
        print(f"Dimensión exportada: {exit_path} | Filas: {len(final_dim)}")

In [29]:
print("1. Cargando catálogos en memoria...")
master_dictionary = catalog_load(CATALOG_PATH)

1. Cargando catálogos en memoria...


In [30]:
print("2. Iniciando extracción en lotes (Chunking)...")
combinations = unique_convin_extract()

2. Iniciando extracción en lotes (Chunking)...
Procesando: ../DATA/Silver/COVID19MEXICO2021/COVID19MEXICO2021.csv...
Procesando: ../DATA/Silver/COVID19MEXICO2022/COVID19MEXICO2022.csv...
Procesando: ../DATA/Silver/COVID19MEXICO2023/COVID19MEXICO2023.csv...
Procesando: ../DATA/Silver/COVID19MEXICO2024/COVID19MEXICO2024.csv...


In [31]:
print("3. Traduciendo y exportando DIMENSIONS...")
process_and_export_dimensions(combinations, master_dictionary)
print("Pipeline finalizado con éxito.")

3. Traduciendo y exportando DIMENSIONS...
Dimensión exportada: dimensiones/DIM_Geografico_informacion_paciente.csv | Filas: 967
Dimensión exportada: dimensiones/DIM_Geografico_residencia.csv | Filas: 2097
Dimensión exportada: dimensiones/DIM_Geografico_Nacionalidad.csv | Filas: 118
Dimensión exportada: dimensiones/DIM_Descripcion_del_paciente.csv | Filas: 405
Dimensión exportada: dimensiones/DIM_Indigena.csv | Filas: 23
Dimensión exportada: dimensiones/DIM_Comorbilidades_Respiratorias.csv | Filas: 115
Dimensión exportada: dimensiones/DIM_Comorbilidades_de_presion.csv | Filas: 114
Dimensión exportada: dimensiones/DIM_Otras_caracteristicas_medicas.csv | Filas: 33
Dimensión exportada: dimensiones/DIM_Ubicacion_de_laboratorio.csv | Filas: 1255
Dimensión exportada: dimensiones/DIM_Antigeno.csv | Filas: 4
Dimensión exportada: dimensiones/DIM_Datos_de_laboratorio.csv | Filas: 13
Pipeline finalizado con éxito.


In [14]:
all_sheets = pd.read_excel('../DICCIONARIOS/240708 Catalogos.xlsx', sheet_name=None)

dim = {
    "DIM_Antigeno": pd.read_csv('dimensiones/DIM_Antigeno.csv'),
    "DIM_Comorbilidades_de_presion": pd.read_csv('dimensiones/DIM_Comorbilidades_de_presion.csv'),
    "DIM_Comorbilidades_Respiratorias": pd.read_csv('dimensiones/DIM_Comorbilidades_Respiratorias.csv'),
    "DIM_Datos_de_laboratorio": pd.read_csv('dimensiones/DIM_Datos_de_laboratorio.csv'),
    "DIM_Descripcion_del_paciente": pd.read_csv('dimensiones/DIM_Descripcion_del_paciente.csv'),
    "DIM_Geografico_informacion_paciente": pd.read_csv('dimensiones/DIM_Geografico_informacion_paciente.csv'),
    "DIM_Geografico_Nacionalidad": pd.read_csv('dimensiones/DIM_Geografico_Nacionalidad.csv'),
    "DIM_GDIM_Geografico_residenciaeo_res": pd.read_csv('dimensiones/DIM_Geografico_residencia.csv'),
    "DIM_Indigena": pd.read_csv('dimensiones/DIM_Indigena.csv'),
    "DIM_Otras_caracteristicas_medicas": pd.read_csv('dimensiones/DIM_Otras_caracteristicas_medicas.csv'),
    "DIM_Ubicacion_de_laboratorio": pd.read_csv('dimensiones/DIM_Ubicacion_de_laboratorio.csv')
}

# =====================================================================
# 2. CONFIGURACIÓN DEL PIPELINE (Reglas de Negocio)
# =====================================================================
configuracion_mapeo = [
    {"df_objetivo": dim["DIM_Antigeno"], "df_catalogo": all_sheets['Catálogo RESULTADO_ANTIGENO'], "indices_columnas": [1, 2]},
    {"df_objetivo": dim["DIM_Comorbilidades_de_presion"], "df_catalogo": all_sheets['Catálogo SI_NO'], "indices_columnas": [1, 2, 3, 4, 5]},
    {"df_objetivo": dim["DIM_Comorbilidades_Respiratorias"], "df_catalogo": all_sheets['Catálogo SI_NO'], "indices_columnas": [1, 2, 3, 4, 5]},
    {"df_objetivo": dim["DIM_Datos_de_laboratorio"], "df_catalogo": all_sheets['Catálogo RESULTADO_LAB'], "indices_columnas": [1, 2, 3, 4]},
    {"df_objetivo": dim["DIM_Descripcion_del_paciente"], "df_catalogo": all_sheets['Catálogo SEXO'], "indices_columnas": [1]},
    {"df_objetivo": dim["DIM_Descripcion_del_paciente"], "df_catalogo": all_sheets['Catálogo TIPO_PACIENTE'], "indices_columnas": [3]},
    {"df_objetivo": dim["DIM_Geografico_informacion_paciente"], "df_catalogo": all_sheets['Catálogo de ENTIDADES'], "indices_columnas": [1, 2]},
    {"df_objetivo": dim["DIM_Geografico_Nacionalidad"], "df_catalogo": all_sheets['Catálogo NACIONALIDAD'], "indices_columnas": [1]},
    {"df_objetivo": dim["DIM_GDIM_Geografico_residenciaeo_res"], "df_catalogo": all_sheets['Catálogo de ENTIDADES'], "indices_columnas": [1]},
    {"df_objetivo": dim["DIM_GDIM_Geografico_residenciaeo_res"], "df_catalogo": all_sheets['Catálogo MUNICIPIOS'], "indices_columnas": [2]},
    {"df_objetivo": dim["DIM_Indigena"], "df_catalogo": all_sheets['Catálogo SI_NO'], "indices_columnas": [1, 2, 3]},
    {"df_objetivo": dim["DIM_Otras_caracteristicas_medicas"], "df_catalogo": all_sheets['Catálogo SI_NO'], "indices_columnas": [1, 2, 3]},
    {"df_objetivo": dim["DIM_Ubicacion_de_laboratorio"], "df_catalogo": all_sheets['Catálogo ORIGEN'], "indices_columnas": [1]},
    {"df_objetivo": dim["DIM_Ubicacion_de_laboratorio"], "df_catalogo": all_sheets['Catálogo SECTOR'], "indices_columnas": [2]},
    {"df_objetivo": dim["DIM_Ubicacion_de_laboratorio"], "df_catalogo": all_sheets['Catálogo SI_NO'], "indices_columnas": [3]}
]

# =====================================================================
# 3. FASE DE TRANSFORMACIÓN (Ejecución del Motor de Mapeo)
# =====================================================================
print("Iniciando mapeo de variables con normalización extrema...")
for numero_tarea, tarea in enumerate(configuracion_mapeo, start=1):
    df_obj = tarea["df_objetivo"]
    df_cat = tarea["df_catalogo"]
    
    col_ref_cat = df_cat.columns[0]
    col_target_cat = df_cat.columns[1]
    
    # 1. NORMALIZACIÓN EXTREMA DEL CATÁLOGO:
    # Convertimos a texto -> Quitamos espacios -> Convertimos a mayúsculas -> Destruimos el ".0" fantasma
    df_cat[col_ref_cat] = df_cat[col_ref_cat].astype(str).str.strip().str.upper().str.replace(r'\.0$', '', regex=True)
    
    # Capa de Calidad de Datos (Manejo de duplicados)
    if df_cat[col_ref_cat].duplicated().any():
        df_cat = df_cat.drop_duplicates(subset=[col_ref_cat], keep='first')
    
    # Construcción del diccionario
    dic_mapeo = df_cat.set_index(col_ref_cat)[col_target_cat]
    
    print(f"\nTAREA #{numero_tarea}: Cruzando con catálogo [{tarea['df_catalogo'].columns[1]}]")
    
    # Mapeo iterativo
    for idx in tarea["indices_columnas"]:
        try:
            col_name = df_obj.columns[idx]
            
            # 2. NORMALIZACIÓN EXTREMA DEL OBJETIVO:
            df_obj[col_name] = df_obj[col_name].astype(str).str.strip().str.upper().str.replace(r'\.0$', '', regex=True)
            
            # --- [NUEVO] DIAGNÓSTICO EN TIEMPO REAL ---
            # Calculamos cuántas coincidencias reales existen antes de mapear
            coincidencias = df_obj[col_name].isin(dic_mapeo.keys()).sum()
            total_filas = len(df_obj)
            porcentaje = (coincidencias / total_filas) * 100 if total_filas > 0 else 0
            print(f" -> Columna [{col_name}]: {coincidencias}/{total_filas} coincidencias ({porcentaje:.1f}%)")
            # ------------------------------------------
            
            # 3. EJECUCIÓN DEL MAPEO: (Retiramos el fillna temporalmente para exponer los nulos si falla)
            df_obj[col_name] = df_obj[col_name].map(dic_mapeo)
            
        except IndexError:
            print(f" 🛑 ERROR DE ÍNDICE: No existe la columna en la posición [{idx}].")
            continue

# =====================================================================
# 4. FASE DE CARGA (Persistencia de Datos - MLOps)
# =====================================================================
directorio_salida = 'dimensiones_map'

print("\n" + "="*50)
print("🚀 INICIANDO GUARDADO EN DIRECTORIO DE SALIDA")
print("="*50)

for nombre_archivo, dataframe_procesado in dim.items():
    archivo_csv = f"{nombre_archivo}.csv"
    ruta_completa = os.path.join(directorio_salida, archivo_csv)
    
    dataframe_procesado.to_csv(ruta_completa, index=False, encoding='utf-8')
    print(f"✅ Guardado exitoso: {ruta_completa}")

print("\n🎯 ETL COMPLETADO: Los archivos transformados están disponibles en 'dimensiones_map/'")

Iniciando mapeo de variables con normalización extrema...

TAREA #1: Cruzando con catálogo [DESCRIPCIÓN]
 -> Columna [TOMA_MUESTRA_ANTIGENO]: 3/4 coincidencias (75.0%)
 -> Columna [RESULTADO_ANTIGENO]: 3/4 coincidencias (75.0%)

TAREA #2: Cruzando con catálogo [DESCRIPCIÓN]
 -> Columna [DIABETES]: 113/114 coincidencias (99.1%)
 -> Columna [INMUSUPR]: 113/114 coincidencias (99.1%)
 -> Columna [HIPERTENSION]: 113/114 coincidencias (99.1%)
 -> Columna [CARDIOVASCULAR]: 113/114 coincidencias (99.1%)
 -> Columna [OBESIDAD]: 113/114 coincidencias (99.1%)

TAREA #3: Cruzando con catálogo [DESCRIPCIÓN]
 -> Columna [INTUBADO]: 114/115 coincidencias (99.1%)
 -> Columna [NEUMONIA]: 114/115 coincidencias (99.1%)
 -> Columna [EPOC]: 114/115 coincidencias (99.1%)
 -> Columna [ASMA]: 114/115 coincidencias (99.1%)
 -> Columna [TABAQUISMO]: 114/115 coincidencias (99.1%)

TAREA #4: Cruzando con catálogo [DESCRIPCIÓN]
 -> Columna [TOMA_MUESTRA_LAB]: 12/13 coincidencias (92.3%)
 -> Columna [RESULTADO_LAB]

In [9]:
A = pd.read_csv('dimensiones/DIM_Antigeno.csv')
print(A.head())
print("---------------")

A = pd.read_csv('dimensiones/DIM_Comorbilidades_de_presion.csv')
print(A.head())
print("---------------")

A = pd.read_csv('dimensiones/DIM_Comorbilidades_Respiratorias.csv')
print(A.head())
print("---------------")
  
A = pd.read_csv('dimensiones/DIM_Datos_de_laboratorio.csv')
print(A.head())
print("---------------")

A = pd.read_csv('dimensiones/DIM_Descripcion_del_paciente.csv')
print(A.head())
print("---------------")

A = pd.read_csv('dimensiones/DIM_Geografico_informacion_paciente.csv')
print(A.head())
print("---------------")

A = pd.read_csv('dimensiones/DIM_Geografico_Nacionalidad.csv')
print(A.head())
print("---------------")

A = pd.read_csv('dimensiones/DIM_Geografico_residencia.csv')
print(A.head())
print("---------------")
 
A = pd.read_csv('dimensiones/DIM_Indigena.csv')
print(A.head())
print("---------------")

A = pd.read_csv('dimensiones/DIM_Otras_caracteristicas_medicas.csv')
print(A.head())
print("---------------")

A = pd.read_csv('dimensiones/DIM_Ubicacion_de_laboratorio.csv')
print(A.head())
print("---------------")


   ID_DIM_ANTIGENO  TOMA_MUESTRA_ANTIGENO  RESULTADO_ANTIGENO
0                1                    1.0                 2.0
1                2                    2.0                97.0
2                3                    1.0                 1.0
3                4                    NaN                 NaN
---------------
   ID_DIM_COMORBILIDADES_DE_PRESION  DIABETES  INMUSUPR  HIPERTENSION  \
0                                 1       2.0       2.0           2.0   
1                                 2       2.0       2.0           2.0   
2                                 3       1.0       2.0           2.0   
3                                 4       2.0       2.0           1.0   
4                                 5       2.0       2.0           2.0   

   CARDIOVASCULAR  OBESIDAD  
0             2.0       2.0  
1             2.0      98.0  
2             2.0       2.0  
3             2.0       1.0  
4             1.0       2.0  
---------------
   ID_DIM_COMORBILIDADES_RESPIRATORIAS 

# USE NEXT CELLS ONLY AFTER RUN "Load_ro_gold" NOTEBOOK

In [5]:
import pyarrow.csv as pv
import pyarrow.parquet as pq
from pathlib import Path

def csv_to_parquet_in_memory(path_csv: str, path_parquet: str):
    table = pv.read_csv(path_csv)
    pq.write_table(table, path_parquet)
    print(f"Conversión exitosa: {path_parquet}")

In [ ]:
csv_to_parquet_in_memory("dimensiones/DIM_Geografico_informacion_paciente.csv", "../DATA/Gold/DIMENSIONES/DIM_Geografico_informacion_paciente.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Comorbilidades_Respiratorias.csv", "../DATA/Gold/DIMENSIONES/DIM_Comorbilidades_Respiratorias.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Datos_de_laboratorio.csv", "../DATA/Gold/DIMENSIONES/DIM_Datos_de_laboratorio.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Otras_caracteristicas_medicas.csv", "../DATA/Gold/DIMENSIONES/DIM_Otras_caracteristicas_medicas.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Antigeno.csv", "../DATA/Gold/DIMENSIONES/DIM_Antigeno.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Geografico_residencia.csv", "../DATA/Gold/DIMENSIONES/DIM_Geografico_residencia.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Ubicacion_de_laboratorio.csv", "../DATA/Gold/DIMENSIONES/DIM_Ubicacion_de_laboratorio.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Comorbilidades_de_presion.csv", "../DATA/Gold/DIMENSIONES/DIM_Comorbilidades_de_presion.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Indigena.csv", "../DATA/Gold/DIMENSIONES/DIM_Indigena.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Descripcion_del_paciente.csv", "../DATA/Gold/DIMENSIONES/DIM_Descripcion_del_paciente.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Geografico_Nacionalidad.csv", "../DATA/Gold/DIMENSIONES/DIM_Geografico_Nacionalidad.parquet")

Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Comorbilidades_Respiratorias.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Datos_de_laboratorio.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Otras_caracteristicas_medicas.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Antigeno.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Geografico_residencia.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Ubicacion_de_laboratorio.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Comorbilidades_de_presion.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Indigena.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Descripcion_del_paciente.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Geografico_Nacionalidad.parquet
